### Problem Statement 4A

4A. Implement the CBOW for document classification using deep learning. 

Steps (for Implementing CBOW for Document Classification):
1. Data Preparation: You need a labeled dataset for document classification. You can use datasets like the IMDb movie reviews dataset (for sentiment classification) or 20 Newsgroups dataset (for topic classification).
Load and Preprocess the data:
- from tensorflow.keras.datasets import imdb # For loading
- Pad sequences to ensure uniform input length
- Convert the IMDB dataset to a list of sentences (each sentence is a list of word)
- Tokenize the text into words.
- Lowercase the text and remove stop words.
- Create a vocabulary from the training data.
2. Define CBOW Model: CBOW learns to predict a target word based on surrounding context words. For document classification, you need to modify the CBOW model to work with document-level contexts. In CBOW, the context consists of words surrounding the target word, but for documents, you will use the embeddings from all the words in the document.
3. Model Architecture:
- Embedding Layer: First, use the CBOW architecture to learn word embeddings from the text corpus.
- Document Representation: After learning the word embeddings, represent a document by averaging the embeddings of all words in the document.
- Fully Connected Layer: Use this document representation as input to a neural network (e.g., a dense layer) for classification.
- Train word2vec model with CBOW (sg=0 for CBOW)
- Convert words to embeddings
- Build the feed forward neural network for document classification
4. Evaluate the Model on test set using loss and accuracy.

In [ ]:
# Optional installation:
# !pip install gensim

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

In [ ]:
# Step 1: Data Preparation - Load and clean IMDb movie reviews
csv_path = "../LP IV Updated Datasets/CBOW/IMDB Dataset.csv"

# Load IMDb dataset (sample 3000 reviews for fast and simple execution)
data = pd.read_csv(csv_path).head(3000)
print(data.head(2))

# Tokenize text into words, lowercase, and simple cleaning
sentences = []
for review in data['review']:
    clean_words = [w.lower() for w in review.split() if w.isalpha() and len(w) > 2]
    sentences.append(clean_words)

# Binary target label (1 = positive, 0 = negative)
y = (data['sentiment'] == 'positive').astype(int).values
print("Total prepared reviews:", len(sentences))

In [ ]:
# Step 2 & 3: Train Word2Vec CBOW model (sg=0) and represent documents
# Train Word2Vec CBOW model (sg=0 specifies Continuous Bag of Words)
cbow_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=0, epochs=10)

# Function to represent each document by averaging word embeddings
def get_document_embedding(word_list):
    valid_vectors = [cbow_model.wv[w] for w in word_list if w in cbow_model.wv]
    if len(valid_vectors) == 0:
        return np.zeros(100)
    return np.mean(valid_vectors, axis=0)

# Convert all reviews to document vectors
X = np.array([get_document_embedding(words) for words in sentences])
print("Document feature matrix shape:", X.shape)

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Step 3 (contd.): Build the Feedforward Neural Network for document classification
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(100,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)

In [ ]:
# Step 4: Evaluate the Model on test set using loss and accuracy
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Plot accuracy and loss
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.show()